In [ ]:
# ============================================================
# CELLULE 1 — Imports & Configuration
# ============================================================
import os
import gc
import json
import time
import random
import warnings
import numpy as np
import torch 
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch_geometric
from torch_geometric.loader import NeighborLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.nn import HeteroConv, GATConv
from torch_geometric.utils import negative_sampling
from torch_geometric.loader import NeighborLoader
from sklearn.metrics import roc_auc_score
 
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
 
warnings.filterwarnings("ignore", category=FutureWarning)
 
# ── Reproductibilité ──────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
 
# ── Chemins Lightning AI ──────────────────────────────────────
DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SNAPSHOT_DIR   = "/kaggle/input/datasets/ayamhiri/temporal-graph-dataset/snapshots"
CHECKPOINT_DIR = "/teamspace/studios/this_studio/checkpoints"
OUTPUT_DIR     = "/kaggle/working/"
 
# ── Hyperparamètres ───────────────────────────────────────────
TRAIN_YEARS    = list(range(2015, 2024))
VAL_YEARS      = [2024]
TEST_YEARS     = [2025]
ALL_YEARS      = sorted(TRAIN_YEARS + VAL_YEARS + TEST_YEARS)
 
NUM_EPOCHS     = 100
CURRENT_EPOCH = 0
LR             = 2e-4
WEIGHT_DECAY   = 1e-4
PATIENCE       = 7
HIDDEN_DIM     = 128
LOGIT_SCALE    = 5.0
BATCH_SIZE     = 8192
NUM_NEIGHBORS  = [40, 40]
RESET_TRAINING = False 
 
for d in [CHECKPOINT_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)
 
BEST_CKPT_PATH     = os.path.join(CHECKPOINT_DIR, "best.pt")
PROGRESS_CKPT_PATH = os.path.join(CHECKPOINT_DIR, "progress.pt")
 
if RESET_TRAINING:
    for p in [BEST_CKPT_PATH, PROGRESS_CKPT_PATH]:
        if os.path.exists(p):
            os.remove(p)
 
print(f"Device      : {DEVICE}")
print(f"Train years : {TRAIN_YEARS}")
print(f"Val years   : {VAL_YEARS}")
print(f"Test years  : {TEST_YEARS}")
print(f"Batch size  : {BATCH_SIZE}")
print(f"Hidden dim  : {HIDDEN_DIM}")
print(f"Reset       : {RESET_TRAINING}")

In [ ]:
# ============================================================
# CELLULE 2 — Modèle THGNN
# ============================================================
 
class SpatialGAT(nn.Module):
    """GAT hétérogène sur un snapshot — 4 relations, heads=2."""
 
    def __init__(self, hidden_dim=128, heads=4, dropout=0.1):
        super().__init__()
        self.conv = HeteroConv({
            ('author', 'writes',     'paper' ): GATConv((-1, -1), hidden_dim, heads=heads,
                concat=False, add_self_loops=False, dropout=dropout),
            ('paper',  'rev_writes', 'author'): GATConv((-1, -1), hidden_dim, heads=heads,
                concat=False, add_self_loops=False, dropout=dropout),
            ('author', 'coauthor',   'author'): GATConv((-1, -1), hidden_dim, heads=heads,
                concat=False, add_self_loops=False, dropout=dropout),
            ('paper',  'cites',      'paper' ): GATConv((-1, -1), hidden_dim, heads=heads,
                concat=False, add_self_loops=False, dropout=dropout),
        }, aggr='sum')
 
        self.norm_paper  = nn.LayerNorm(hidden_dim)
        self.norm_author = nn.LayerNorm(hidden_dim)
 
    def forward(self, x_dict, edge_index_dict):
        out = self.conv(x_dict, edge_index_dict)
        out['paper']  = self.norm_paper(out['paper'])
        out['author'] = self.norm_author(out['author'])
        return out
 
 
class THGNN(nn.Module):
    """
    Temporal Heterogeneous GNN — GAT spatial + GRU step-by-step.
 
    Traite UN batch à la fois et reçoit l'état GRU précédent
    (prior_state) depuis la state bank externe du training loop.
 
    Changements v2 :
    - hidden_dim 64 → 128
    - heads 4 → 2 (évite OOM sur grands snapshots)
    - Masque is_new : distingue auteurs existants vs nouveaux
    """
 
    def __init__(self, paper_dim=384, author_dim=32, hidden_dim=128):
        super().__init__()
        self.hidden_dim = hidden_dim
 
        self.paper_bn    = nn.BatchNorm1d(paper_dim)
        self.author_bn   = nn.BatchNorm1d(author_dim)
        self.paper_proj  = nn.Linear(paper_dim,  hidden_dim)
        self.author_proj = nn.Linear(author_dim, hidden_dim)
 
        self.spatial_gat     = SpatialGAT(hidden_dim, heads=2)
        self.author_gru_cell = nn.GRUCell(hidden_dim, hidden_dim)
 
        self.norm_out_author = nn.LayerNorm(hidden_dim)
        self.norm_out_paper  = nn.LayerNorm(hidden_dim)
 
    def forward(self, data, prior_state=None):
        """
        data        : HeteroData du batch courant (sur GPU)
        prior_state : (N_authors_in_batch, hidden_dim) sur GPU
                      États h_{t-1} extraits de la state bank.
        """
        dev = data['author'].x.device
 
        # ── 1. Projection features ────────────────────────────
        x_author_raw = data['author'].x
        x_paper_raw  = data['paper'].x
 
        x_author = self.author_proj(
            self.author_bn(x_author_raw) if x_author_raw.size(0) > 1 else x_author_raw
        )
        x_paper = self.paper_proj(
            self.paper_bn(x_paper_raw) if x_paper_raw.size(0) > 1 else x_paper_raw
        )
 
        # ── 2. GAT spatial ────────────────────────────────────
        out        = self.spatial_gat({'author': x_author, 'paper': x_paper},
                                      data.edge_index_dict)
        gat_author = out['author']
        gat_paper  = out['paper']
 
        # ── 3. GRU avec masque is_new ─────────────────────────
        if prior_state is None:
            h_prev = torch.zeros(gat_author.size(0), self.hidden_dim, device=dev)
        else:
            h_prev = prior_state.to(dev)
            # Auteurs jamais vus → état nul propre
            is_new         = (h_prev.abs().sum(dim=-1) < 1e-9)
            h_prev         = h_prev.clone()
            h_prev[is_new] = 0.0
 
        z_author = self.author_gru_cell(gat_author, h_prev)
 
        # ── 4. Normalisation & L2 ─────────────────────────────
        z_author = F.normalize(self.norm_out_author(z_author), dim=-1)
        z_paper  = F.normalize(self.norm_out_paper(gat_paper),  dim=-1)
 
        return z_author, z_paper
 
 

In [ ]:
# ============================================================
# CELLULE 3 — Chargement snapshots & mapping global
# ============================================================
 
def load_snapshots(years, snapshot_dir, device="cpu"):
    snaps = {}
    for y in years:
        path = os.path.join(snapshot_dir, f"snapshot_{y}.pt")
        if os.path.exists(path):
            snaps[y] = torch.load(path, map_location=device, weights_only=False)
        else:
            print(f"  ⚠️  Snapshot manquant : {path}")
    return snaps
 
snap_by_year = load_snapshots(ALL_YEARS, SNAPSHOT_DIR)
print(f"Snapshots chargés : {len(snap_by_year)}/{len(ALL_YEARS)}")
 
print("🔍 Calcul du mapping global...")
global_author_ids = set()
for y in ALL_YEARS:
    path = os.path.join(SNAPSHOT_DIR, f"snapshot_{y}.pt")
    if os.path.exists(path):
        tmp = torch.load(path, map_location="cpu", weights_only=False)
        global_author_ids.update(tmp['author'].node_id.tolist())
        del tmp
        gc.collect()
 
global_author_ids    = sorted(list(global_author_ids))
author_id_to_compact = {int(aid): idx for idx, aid in enumerate(global_author_ids)}
NUM_AUTHORS_GLOBAL   = len(global_author_ids)
print(f"Auteurs globaux : {NUM_AUTHORS_GLOBAL:,}")
print(f"State bank RAM  : {NUM_AUTHORS_GLOBAL * HIDDEN_DIM * 4 / 1e6:.0f} MB")
 

In [ ]:
# ============================================================
# CELLULE 4 — LOSS & METRICS (FINAL CLEAN)
# ============================================================

TRAIN_PHASE = 1  # sera contrôlée uniquement par CELLULE 7

def init_author_state_bank(num_authors, hidden_dim, dtype=torch.float32):
    return torch.zeros(num_authors, hidden_dim, dtype=dtype)


def safe_negative_sampling(edge_index, num_src, num_dst, num_neg):
    if edge_index.numel() == 0 or num_src <= 1 or num_dst <= 1 or num_neg == 0:
        return None

    return negative_sampling(
        edge_index=edge_index,
        num_nodes=(num_src, num_dst) if num_src != num_dst else num_src,
        num_neg_samples=num_neg,
        method="sparse"
    )


def relation_bce_loss(edge_index, emb_src, emb_dst, scale=LOGIT_SCALE):
    if edge_index.numel() == 0:
        return None, None, None

    pos_logits = scale * (emb_src[edge_index[0]] * emb_dst[edge_index[1]]).sum(dim=-1)

    neg_ei = safe_negative_sampling(
        edge_index, emb_src.size(0), emb_dst.size(0), edge_index.size(1)
    )

    if neg_ei is None or neg_ei.numel() == 0:
        return None, None, None

    neg_logits = scale * (emb_src[neg_ei[0]] * emb_dst[neg_ei[1]]).sum(dim=-1)

    logits = torch.cat([pos_logits, neg_logits])
    labels = torch.cat([torch.ones_like(pos_logits), torch.zeros_like(neg_logits)])

    loss = F.binary_cross_entropy_with_logits(logits, labels)

    if torch.isnan(loss):
        return None, None, None

    return (
        loss,
        logits.detach().cpu().numpy(),
        labels.detach().cpu().numpy()
    )


def compute_vicreg(z):
    z2 = z + 0.01 * torch.randn_like(z)

    sim_loss = F.mse_loss(z, z2)

    std = torch.sqrt(z.var(dim=0) + 1e-4)
    var_loss = torch.mean(F.relu(1 - std))

    zc = z - z.mean(dim=0)
    cov = (zc.T @ zc) / (z.size(0) - 1)
    cov_loss = (cov.fill_diagonal_(0)).pow(2).sum() / z.size(1)

    return sim_loss + var_loss + cov_loss


def temporal_loss_fn(z_author, author_state_bank_cpu, global_idx):
    z_prev = author_state_bank_cpu[global_idx].to(DEVICE)
    mask = (z_prev.abs().sum(dim=-1) > 1e-9)

    if mask.sum() == 0:
        return torch.tensor(0.0, device=DEVICE)

    return F.mse_loss(z_author[mask], z_prev[mask].detach())


def compute_loss_and_metrics(z_author, z_paper, data,
                             author_state_bank_cpu, global_idx):

    data = data.to(DEVICE)

    relations = [
        (('author','writes','paper'), z_author, z_paper),
        (('author','coauthor','author'), z_author, z_author),
        (('paper','cites','paper'), z_paper, z_paper),
    ]

    bce_losses, logits_list, labels_list = [], [], []

    for rel, src, dst in relations:
        if rel not in data.edge_types:
            continue

        ei = data[rel].edge_index
        if ei.numel() == 0:
            continue

        loss_r, logits_np, labels_np = relation_bce_loss(ei, src, dst)

        if loss_r is None:
            continue

        bce_losses.append(loss_r)
        logits_list.append(logits_np)
        labels_list.append(labels_np)

    loss_bce = torch.stack(bce_losses).mean() if bce_losses else torch.tensor(0.0, device=DEVICE)

    loss_vicreg = compute_vicreg(z_author)
    loss_temp = temporal_loss_fn(z_author, author_state_bank_cpu, global_idx)

    global TRAIN_PHASE

    if TRAIN_PHASE == 1:
        loss_total = 1.0*loss_bce + 0.7*loss_vicreg + 0.1*loss_temp
    else:
        loss_total = 0.1*loss_bce + 1.0*loss_vicreg + 0.2*loss_temp

    if logits_list:
        logits = np.concatenate(logits_list)
        labels = np.concatenate(labels_list)
        probs = 1/(1+np.exp(-logits))

        auc = roc_auc_score(labels, probs)
        acc = float(((probs >= 0.5).astype(int) == labels.astype(int)).mean())
    else:
        auc, acc = 0.5, 0.5

    return loss_total, loss_bce.item(), loss_vicreg.item(), loss_temp.item(), auc, acc

In [ ]:
# ============================================================
# CELLULE 5 — Helpers & forward_year avec batching
# ============================================================
 
def gpu_mem_mb():
    return torch.cuda.memory_allocated() / 1e6 if DEVICE.type == "cuda" else 0.0
 
def gpu_reserved_mb():
    return torch.cuda.memory_reserved() / 1e6 if DEVICE.type == "cuda" else 0.0
 
def ram_mb():
    try:
        import psutil
        return psutil.Process(os.getpid()).memory_info().rss / 1e6
    except ImportError:
        return 0.0
 
 
def forward_year(model, year, author_state_bank_cpu, train=True):
    path = os.path.join(SNAPSHOT_DIR, f"snapshot_{year}.pt")
    data = torch.load(path, map_location="cpu", weights_only=False)

    # Mapping global
    node_ids = data['author'].node_id.tolist()
    compact  = [author_id_to_compact[int(aid)] for aid in node_ids]
    g_idx    = torch.tensor(compact, dtype=torch.long)
    data['author'].global_idx = g_idx

    # ── NeighborLoader ─────────────────────────────────────────
    loader = NeighborLoader(
        data,
        num_neighbors=NUM_NEIGHBORS,
        input_nodes=('author', None),
        batch_size=BATCH_SIZE,
        shuffle=train,
        num_workers=4,
    )

    # 🔥 UPDATED (plus de NCE)
    batch_losses, batch_bces = [], []
    batch_vicregs, batch_temps = [], []
    batch_aucs, batch_accs = [], []

    model.train() if train else model.eval()

    for batch in loader:
        g_idx_batch = batch['author'].global_idx
        prior_state = author_state_bank_cpu[g_idx_batch].to(DEVICE)
        batch_gpu   = batch.to(DEVICE)

        if train:
            optimizer.zero_grad(set_to_none=True)

            z_author, z_paper = model(batch_gpu, prior_state)

            loss_total, loss_bce, loss_vicreg, loss_temp, auc, acc = compute_loss_and_metrics(
                z_author, z_paper, batch_gpu, author_state_bank_cpu, g_idx_batch
            )

            loss_total.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        else:
            with torch.no_grad():
                z_author, z_paper = model(batch_gpu, prior_state)

                loss_total, loss_bce, loss_vicreg, loss_temp, auc, acc = compute_loss_and_metrics(
                    z_author, z_paper, batch_gpu, author_state_bank_cpu, g_idx_batch
                )

        # ✅ update state bank
        author_state_bank_cpu[g_idx_batch] = z_author.detach().cpu()
        

        # 🔥 UPDATED tracking
        batch_losses.append(float(loss_total.item()))
        batch_bces.append(loss_bce)
        batch_vicregs.append(loss_vicreg)
        batch_temps.append(loss_temp)
        batch_aucs.append(auc)
        batch_accs.append(acc)

        
        del batch_gpu, prior_state, z_author, z_paper, loss_total

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    gc.collect()
     # ============================================================
    # 🔥 PRINT DÉTAILLÉ PAR YEAR (TON FORMAT EXACT)
    # ============================================================
    print(
        f"Epoch {CURRENT_EPOCH+1:03d}/{NUM_EPOCHS} | {year} | "
        f"total={float(np.mean(batch_losses)):.4f} | "
        f"bce={float(np.mean(batch_bces)):.4f} | "
        f"vicreg={float(np.mean(batch_vicregs)):.4f} | "
        f"temp={float(np.mean(batch_temps)):.4f} | "
        f"auc={float(np.mean(batch_aucs)):.4f} | "
        f"GPU={gpu_mem_mb():.0f}MB"
    )

    return (
        author_state_bank_cpu,
        float(np.mean(batch_losses)),
        float(np.mean(batch_bces)),
        float(np.mean(batch_vicregs)),   # 🔥 remplacé
        float(np.mean(batch_temps)),
        float(np.mean(batch_aucs)),
        float(np.mean(batch_accs)),
    )
 
 

In [ ]:
# ============================================================
# CELLULE 6 — Checkpoint helpers & instanciation modèle (FIX)
# ============================================================

def save_checkpoint(path, payload):
    tmp = path + ".tmp"
    torch.save(payload, tmp)
    os.replace(tmp, path)


def build_checkpoint(epoch, year, next_year_idx, best_val_loss, history, bank_cpu):
    return {
        "epoch": epoch,
        "year": year,
        "next_year_idx": next_year_idx,
        "best_val_loss": best_val_loss,
        "history": history,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "author_state_bank_cpu": bank_cpu,
        "global_author_ids": global_author_ids,

        # ✅ FIX CRITIQUE
        "train_phase": TRAIN_PHASE,

        "config": {
            "epochs": NUM_EPOCHS,
            "lr": LR,
            "weight_decay": WEIGHT_DECAY,
            "patience": PATIENCE,
            "hidden_dim": HIDDEN_DIM,
            "batch_size": BATCH_SIZE,
            "num_neighbors": NUM_NEIGHBORS,
            "logit_scale": LOGIT_SCALE,
        }
    }


def load_checkpoint(path):
    return torch.load(path, map_location="cpu", weights_only=False)


# MODEL INIT
snap0 = snap_by_year[ALL_YEARS[0]]

model = THGNN(
    paper_dim=snap0['paper'].x.shape[1],
    author_dim=snap0['author'].x.shape[1],
    hidden_dim=HIDDEN_DIM,
).to(DEVICE)

with torch.no_grad():
    model(snap0.to(DEVICE))

optimizer = Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

print(f"✅ Modèle THGNN | Paramètres : {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ============================================================
# CELLULE 7 — TRAIN LOOP FINAL (2 PHASES CORRECT + FULL LOG)
# ============================================================

history = {
    # epoch-level
    "train_loss": [], "val_loss": [],
    "train_bce": [], "val_bce": [],
    "train_vicreg": [], "val_vicreg": [],
    "train_temp": [], "val_temp": [],
    "train_auc": [], "val_auc": [],
    "train_acc": [], "val_acc": [],

    # system
    "lr": [],
    "epoch_time_s": [],
    "gpu_mb": [],
    "ram_mb": [],
    "phase": [],

    # year-level
    "year": [],
    "year_train_loss": [],
    "year_train_bce": [],
    "year_train_vicreg": [],
    "year_train_temp": [],
    "year_train_auc": [],
    "year_train_acc": []
}

# ============================================================
# INIT
# ============================================================
start_epoch = 0
best_val_loss = float("inf")

# ✅ GLOBAL BEST (NEW FIX)
best_global_val_loss = float("inf")

CURRENT_EPOCH = 0
TRAIN_PHASE = 1

author_state_bank_cpu = init_author_state_bank(NUM_AUTHORS_GLOBAL, HIDDEN_DIM)

t_total = time.time()

patience_counter = 0
phase2_patience_counter = 0


# ============================================================
# LOAD CHECKPOINT
# ============================================================
if not RESET_TRAINING and os.path.exists(PROGRESS_CKPT_PATH):

    ckpt = load_checkpoint(PROGRESS_CKPT_PATH)

    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])

    start_epoch = int(ckpt.get("epoch", 0))
    best_val_loss = float(ckpt.get("best_val_loss", float("inf")))
    author_state_bank_cpu = ckpt.get("author_state_bank_cpu", author_state_bank_cpu)

    TRAIN_PHASE = ckpt.get("train_phase", 1)

    print(f"♻️ Resume epoch={start_epoch} | phase={TRAIN_PHASE}")

else:
    print("🆕 Training from scratch")


# ============================================================
# TRAIN LOOP
# ============================================================
for epoch in range(start_epoch, NUM_EPOCHS):

    CURRENT_EPOCH = epoch
    t_epoch = time.time()

    warmup_ratio = 0.3
    warmup_done = (CURRENT_EPOCH / NUM_EPOCHS) >= warmup_ratio

    # ========================================================
    # TRAIN PER YEAR
    # ========================================================
    year_losses, year_bces, year_vicregs, year_temps = [], [], [], []
    year_aucs, year_accs = [], []

    for year in TRAIN_YEARS:

        author_state_bank_cpu, y_loss, y_bce, y_vicreg, y_temp, y_auc, y_acc = forward_year(
            model, year, author_state_bank_cpu, train=True
        )

        year_losses.append(y_loss)
        year_bces.append(y_bce)
        year_vicregs.append(y_vicreg)
        year_temps.append(y_temp)
        year_aucs.append(y_auc)
        year_accs.append(y_acc)

        history["year"].append(year)
        history["year_train_loss"].append(y_loss)
        history["year_train_bce"].append(y_bce)
        history["year_train_vicreg"].append(y_vicreg)
        history["year_train_temp"].append(y_temp)
        history["year_train_auc"].append(y_auc)
        history["year_train_acc"].append(y_acc)

    # ========================================================
    # AGGREGATION
    # ========================================================
    train_loss   = float(np.mean(year_losses))
    train_bce    = float(np.mean(year_bces))
    train_vicreg = float(np.mean(year_vicregs))
    train_temp   = float(np.mean(year_temps))
    train_auc    = float(np.mean(year_aucs))
    train_acc    = float(np.mean(year_accs))

    # ========================================================
    # VALIDATION
    # ========================================================
    val_bank = author_state_bank_cpu.clone()

    val_bank, val_loss, val_bce, val_vicreg, val_temp, val_auc, val_acc = forward_year(
        model, VAL_YEARS[-1], val_bank, train=False
    )

    scheduler.step(val_loss)

    lr = optimizer.param_groups[0]["lr"]
    epoch_time = time.time() - t_epoch

    # ========================================================
    # HISTORY (EPOCH)
    # ========================================================
    history["train_loss"].append(train_loss)
    history["train_bce"].append(train_bce)
    history["train_vicreg"].append(train_vicreg)
    history["train_temp"].append(train_temp)
    history["train_auc"].append(train_auc)
    history["train_acc"].append(train_acc)

    history["val_loss"].append(val_loss)
    history["val_bce"].append(val_bce)
    history["val_vicreg"].append(val_vicreg)
    history["val_temp"].append(val_temp)
    history["val_auc"].append(val_auc)
    history["val_acc"].append(val_acc)

    history["lr"].append(lr)
    history["epoch_time_s"].append(epoch_time)
    history["gpu_mb"].append(gpu_mem_mb())
    history["ram_mb"].append(ram_mb())
    history["phase"].append(TRAIN_PHASE)

    # ========================================================
    # PRINT
    # ========================================================
    print("\n" + "─"*85)
    print(f"✅ Epoch {epoch+1}/{NUM_EPOCHS} | Phase {TRAIN_PHASE} | lr={lr:.2e} | time={epoch_time:.1f}s")
    print(f"GPU={gpu_mem_mb():.0f}MB | RAM={ram_mb():.0f}MB")

    print(f"TRAIN → loss={train_loss:.4f} | bce={train_bce:.4f} | vicreg={train_vicreg:.4f} | temp={train_temp:.4f} | auc={train_auc:.4f} | acc={train_acc:.4f}")
    print(f"VAL   → loss={val_loss:.4f} | bce={val_bce:.4f} | vicreg={val_vicreg:.4f} | temp={val_temp:.4f} | auc={val_auc:.4f} | acc={val_acc:.4f}")
    print("─"*85)

    # ========================================================
    # PHASE LOGIC
    # ========================================================
    if TRAIN_PHASE == 1:

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE or warmup_done:
            print("🟡 SWITCH → Phase 2")
            TRAIN_PHASE = 2
            patience_counter = 0

    else:

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            phase2_patience_counter = 0
        else:
            phase2_patience_counter += 1

        if phase2_patience_counter >= PATIENCE:
            print("🛑 EARLY STOP FINAL")
            break

    # ========================================================
    # ✅ GLOBAL BEST CHECKPOINT (FIX)
    # ========================================================
    if val_loss < best_global_val_loss:

        best_global_val_loss = val_loss

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "author_state_bank_cpu": author_state_bank_cpu,
                "epoch": epoch,
                "train_phase": TRAIN_PHASE,
                "best_val_loss": best_global_val_loss
            },
            "checkpoints/best.pt"
        )

        print(f"🏆 NEW GLOBAL BEST → epoch {epoch+1} | val_loss={val_loss:.4f}")

    # ========================================================
    # PROGRESS CHECKPOINT
    # ========================================================
    save_checkpoint(
        PROGRESS_CKPT_PATH,
        build_checkpoint(
            epoch=epoch,
            year=None,
            next_year_idx=0,
            best_val_loss=best_val_loss,
            history=history,
            bank_cpu=author_state_bank_cpu
        )
    )

# ============================================================
# END
# ============================================================
print(f"\n⏱️ Total training: {time.time()-t_total:.1f}s")

In [ ]:
# ============================================================
# CELLULE — POST TRAINING ANALYSIS
# ============================================================

import json
import numpy as np
import matplotlib.pyplot as plt

metrics_path = os.path.join(OUTPUT_DIR, "training_metrics.json")
plots_dir = os.path.join(OUTPUT_DIR, "plots")
os.makedirs(plots_dir, exist_ok=True)

# ============================================================
# 1. SAVE METRICS TO FILE
# ============================================================
with open(metrics_path, "w") as f:
    json.dump(history, f)

print(f"✅ Metrics saved → {metrics_path}")

# ============================================================
# 2. BUILD X AXIS
# ============================================================
epochs = np.arange(1, len(history["train_loss"]) + 1)

# ============================================================
# 3. LOSS CURVES
# ============================================================
plt.figure(figsize=(10,5))
plt.plot(epochs, history["train_loss"], label="Train Loss")
plt.plot(epochs, history["val_loss"], label="Val Loss")
plt.title("Total Loss")
plt.legend()
plt.grid()
plt.savefig(os.path.join(plots_dir, "loss.png"))
plt.show()

# ============================================================
# 4. BCE + VICREG
# ============================================================
plt.figure(figsize=(10,5))
plt.plot(epochs, history["train_bce"], label="Train BCE")
plt.plot(epochs, history["val_bce"], label="Val BCE")
plt.title("BCE Loss")
plt.legend()
plt.grid()
plt.savefig(os.path.join(plots_dir, "bce.png"))
plt.show()

plt.figure(figsize=(10,5))
plt.plot(epochs, history["train_vicreg"], label="Train VICReg")
plt.plot(epochs, history["val_vicreg"], label="Val VICReg")
plt.title("VICReg Loss")
plt.legend()
plt.grid()
plt.savefig(os.path.join(plots_dir, "vicreg.png"))
plt.show()

# ============================================================
# 5. AUC / ACC
# ============================================================
plt.figure(figsize=(10,5))
plt.plot(epochs, history["train_auc"], label="Train AUC")
plt.plot(epochs, history["val_auc"], label="Val AUC")
plt.title("AUC")
plt.legend()
plt.grid()
plt.savefig(os.path.join(plots_dir, "auc.png"))
plt.show()

plt.figure(figsize=(10,5))
plt.plot(epochs, history["train_acc"], label="Train Acc")
plt.plot(epochs, history["val_acc"], label="Val Acc")
plt.title("Accuracy")
plt.legend()
plt.grid()
plt.savefig(os.path.join(plots_dir, "acc.png"))
plt.show()

# ============================================================
# 6. LEARNING RATE
# ============================================================
plt.figure(figsize=(10,5))
plt.plot(epochs, history["lr"], label="LR")
plt.title("Learning Rate Schedule")
plt.legend()
plt.grid()
plt.savefig(os.path.join(plots_dir, "lr.png"))
plt.show()

# ============================================================
# 7. GPU + RAM USAGE
# ============================================================
plt.figure(figsize=(10,5))
plt.plot(epochs, history["gpu_mb"], label="GPU MB")
plt.plot(epochs, history["ram_mb"], label="RAM MB")
plt.title("Memory Usage")
plt.legend()
plt.grid()
plt.savefig(os.path.join(plots_dir, "memory.png"))
plt.show()

# ============================================================
# 8. PHASE VISUALIZATION
# ============================================================
if "phase" in history and len(history["phase"]) > 0:

    plt.figure(figsize=(10,3))
    plt.plot(epochs, history["phase"], label="Phase (1/2)")
    plt.title("Training Phase Switch")
    plt.yticks([1,2])
    plt.grid()
    plt.savefig(os.path.join(plots_dir, "phase.png"))
    plt.show()

# ============================================================
# DONE
# ============================================================
print("✅ All plots generated →", plots_dir)

In [ ]:
# ============================================================
# CELLULE 8 — CLUSTERING 2024 & 2025 (INFÉRENCE POST-TRAIN)
# ============================================================

import os, gc, json, time, warnings
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter
warnings.filterwarnings("ignore")

try:
    import umap
except ImportError:
    os.system("pip install umap-learn -q")
    import umap

try:
    import hdbscan
except ImportError:
    os.system("pip install hdbscan -q")
    import hdbscan

try:
    import networkx as nx
except ImportError:
    os.system("pip install networkx -q")
    import networkx as nx

from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
)

# ── Config ─────────────────────────────────────────────────────
CLUSTER_YEARS      = [2024, 2025]
MIN_CLUSTER_SIZE   = 300
MIN_SAMPLES        = 30
UMAP_N_NEIGHBORS   = 15
UMAP_N_COMPONENTS  = 2
UMAP_MIN_DIST      = 0.05
UMAP_METRIC        = "cosine"

CKPT_PATH          = "/kaggle/input/datasets/ayamhiri/best-modele"
SNAPSHOT_DIR       = "/kaggle/input/datasets/ayamhiri/temporal-graph-dataset/snapshots"
CLUSTER_OUTPUT_DIR = "/kaggle/working/clustering"
os.makedirs(CLUSTER_OUTPUT_DIR, exist_ok=True)

# ── 1. Chargement checkpoint ────────────────────────────────────
print(f"📦 Chargement checkpoint : {CKPT_PATH}")
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)

snap0 = snap_by_year[ALL_YEARS[0]]
model_clust = THGNN(
    paper_dim  = ckpt["config"].get("paper_dim",  snap0['paper'].x.shape[1]),
    author_dim = ckpt["config"].get("author_dim", snap0['author'].x.shape[1]),
    hidden_dim = ckpt["config"].get("hidden_dim", HIDDEN_DIM),
).to(DEVICE)

model_clust.load_state_dict(ckpt["model_state_dict"])
model_clust.eval()
print(f"✅ Modèle chargé | epoch={ckpt.get('epoch','?')} | "
      f"val_loss={ckpt.get('best_val_loss','?')}")

author_state_bank_clust = ckpt.get(
    "author_state_bank_cpu",
    init_author_state_bank(NUM_AUTHORS_GLOBAL, HIDDEN_DIM)
)
print(f"✅ State bank : {author_state_bank_clust.shape}  "
      f"({author_state_bank_clust.shape[0]:,} auteurs × {author_state_bank_clust.shape[1]} dims)")


# ── 2. Helpers ─────────────────────────────────────────────────
def make_palette(n):
    base = plt.cm.get_cmap("tab20", max(n, 20))
    return [base(i % 20) for i in range(n)]


def build_community_graph(labels, z_np, top_k=8):
    unique = [c for c in np.unique(labels) if c != -1]
    if len(unique) < 2:
        return None, {}
    centroids = {}
    for c in unique:
        mask     = labels == c
        centroid = z_np[mask].mean(axis=0)
        norm     = np.linalg.norm(centroid)
        centroids[c] = centroid / (norm + 1e-9)
    G = nx.Graph()
    for c in unique:
        G.add_node(c, size=int((labels == c).sum()))
    ids     = list(unique)
    mat     = np.stack([centroids[c] for c in ids])
    sim_mat = mat @ mat.T
    for i in range(len(ids)):
        row = [(sim_mat[i, j], ids[j]) for j in range(len(ids)) if j != i]
        row.sort(reverse=True)
        for sim_val, j_id in row[:top_k]:
            if sim_val > 0.3:
                G.add_edge(ids[i], j_id, weight=float(sim_val))
    communities = {}
    for node in G.nodes():
        communities[int(node)] = {
            "cluster_id": int(node),
            "size":        int((labels == node).sum()),
            "top_neighbors": [
                {"cluster": int(nb), "similarity": float(G[node][nb]["weight"])}
                for nb in sorted(G[node],
                                 key=lambda x: G[node][x]["weight"],
                                 reverse=True)[:5]
            ] if G.degree(node) > 0 else [],
        }
    return G, communities


# ── 3. Inférence + clustering par année ────────────────────────
def cluster_year_inference(year, model, bank_cpu):
    print(f"\n{'='*70}")
    print(f"  📌 Clustering — Année {year}  (inférence only, modèle figé)")
    print(f"{'='*70}")

    snap_path = os.path.join(SNAPSHOT_DIR, f"snapshot_{year}.pt")
    if not os.path.exists(snap_path):
        print(f"  ⚠️  Snapshot manquant pour {year}, skipped.")
        return None

    data     = torch.load(snap_path, map_location="cpu", weights_only=False)
    node_ids = data['author'].node_id.tolist()
    compact  = [author_id_to_compact[int(aid)] for aid in node_ids]
    g_idx    = torch.tensor(compact, dtype=torch.long)
    data['author'].global_idx = g_idx

    loader = NeighborLoader(
        data,
        num_neighbors = NUM_NEIGHBORS,
        input_nodes   = ('author', None),
        batch_size    = BATCH_SIZE,
        shuffle       = False,
        num_workers   = 0,
    )

    model.eval()
    n_authors      = len(node_ids)
    all_z          = []
    all_g_idx_list = []

    print(f"  ▶ Inférence sur {n_authors:,} auteurs "
          f"(NeighborLoader batch={BATCH_SIZE}) ...")
    t0 = time.time()

    with torch.no_grad():
        for i, batch in enumerate(loader):
            n_seeds         = batch['author'].batch_size
            g_idx_batch_all = batch['author'].global_idx
            prior_state     = bank_cpu[g_idx_batch_all].to(DEVICE)
            batch_gpu       = batch.to(DEVICE)
            z_author, _     = model(batch_gpu, prior_state)

            all_z.append(z_author[:n_seeds].cpu())
            all_g_idx_list.append(g_idx_batch_all[:n_seeds])

            del batch_gpu, prior_state, z_author
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

            if i % 5 == 0:
                done = min((i + 1) * BATCH_SIZE, n_authors)
                print(f"    batch {i+1} | {done:,}/{n_authors:,} "
                      f"({100*done/n_authors:.0f}%) | GPU={gpu_mem_mb():.0f}MB")

    z_cpu     = torch.cat(all_z,          dim=0)
    g_idx_all = torch.cat(all_g_idx_list, dim=0)
    gc.collect()
    print(f"  ✅ Inférence terminée en {time.time()-t0:.1f}s → shape {list(z_cpu.shape)}")

    # ── ✅ Mise à jour state bank ───────────────────────────────
    bank_cpu[g_idx_all] = z_cpu
    print(f"  ✅ State bank mise à jour ({g_idx_all.shape[0]:,} auteurs → prêt pour année suivante)")

    # ── Sauvegarde embeddings bruts ─────────────────────────────
    emb_path = os.path.join(CLUSTER_OUTPUT_DIR, f"embeddings_{year}.pt")
    torch.save(
        {
            "year":       year,
            "embeddings": z_cpu,
            "node_ids":   torch.tensor(node_ids, dtype=torch.long),
            "global_idx": g_idx_all,
            "hidden_dim": z_cpu.shape[1],
            "n_authors":  z_cpu.shape[0],
        },
        emb_path
    )
    print(f"  💾 Embeddings bruts  → {emb_path}  "
          f"[shape={list(z_cpu.shape)}, dtype={z_cpu.dtype}]")

    # ── HDBSCAN sur espace haute dimension ──────────────────────
    z_np = z_cpu.numpy().astype(np.float32)
    print(f"  ▶ HDBSCAN (min_cluster_size={MIN_CLUSTER_SIZE}, "
          f"min_samples={MIN_SAMPLES}) sur embeddings {z_np.shape[1]}-dim ...")
    t0 = time.time()

    clusterer = hdbscan.HDBSCAN(
        min_cluster_size         = MIN_CLUSTER_SIZE,
        min_samples              = MIN_SAMPLES,
        metric                   = "cosine",
        cluster_selection_method = "eom",
        prediction_data          = True,
        core_dist_n_jobs         = -1,
    )
    labels     = clusterer.fit_predict(z_np)
    n_clusters = int(len(set(labels)) - (1 if -1 in labels else 0))
    noise      = int((labels == -1).sum())
    noise_pct  = 100.0 * noise / len(labels)
    print(f"  ✅ HDBSCAN terminé en {time.time()-t0:.1f}s")
    print(f"     → {n_clusters} clusters | bruit={noise} ({noise_pct:.1f}%)")

    # ── Métriques sur espace original ───────────────────────────
    mask_valid = labels != -1
    if mask_valid.sum() > 1 and n_clusters > 1:
        sil = silhouette_score(
            z_np[mask_valid], labels[mask_valid],
            metric      = "cosine",
            sample_size = min(10_000, int(mask_valid.sum()))
        )
        db  = davies_bouldin_score(z_np[mask_valid], labels[mask_valid])
        ch  = calinski_harabasz_score(z_np[mask_valid], labels[mask_valid])
    else:
        sil, db, ch = 0.0, 0.0, 0.0
    print(f"     Silhouette={sil:.4f} | Davies-Bouldin={db:.4f} | "
          f"Calinski-Harabasz={ch:.2f}")

    # ── UMAP uniquement pour visualisation ──────────────────────
    print(f"  ▶ UMAP  (n_neighbors={UMAP_N_NEIGHBORS}, metric={UMAP_METRIC}) "
          f"[visualisation uniquement] ...")
    t0 = time.time()
    reducer = umap.UMAP(
        n_neighbors  = UMAP_N_NEIGHBORS,
        n_components = UMAP_N_COMPONENTS,
        min_dist     = UMAP_MIN_DIST,
        metric       = UMAP_METRIC,
        random_state = SEED,
        low_memory   = True,
        verbose      = False,
    )
    z_2d = reducer.fit_transform(z_np)
    print(f"  ✅ UMAP terminé en {time.time()-t0:.1f}s → shape {z_2d.shape}")

    # ── Communautés ─────────────────────────────────────────────
    G_comm, communities = build_community_graph(labels, z_np, top_k=8)
    print(f"  ✅ {len(communities)} communautés construites")

    # ── Sauvegarde NPZ ──────────────────────────────────────────
    npz_path = os.path.join(CLUSTER_OUTPUT_DIR, f"cluster_{year}.npz")
    np.savez_compressed(
        npz_path,
        z_2d     = z_2d,
        labels   = labels,
        node_ids = np.array(node_ids, dtype=np.int64),
    )
    print(f"  💾 UMAP + labels     → {npz_path}")

    # ── JSON métriques ──────────────────────────────────────────
    metrics = {
        "year":              year,
        "n_clusters":        n_clusters,
        "noise":             noise,
        "noise_pct":         round(noise_pct, 2),
        "silhouette":        float(sil),
        "davies_bouldin":    float(db),
        "calinski_harabasz": float(ch),
        "min_cluster_size":  MIN_CLUSTER_SIZE,
        "min_samples":       MIN_SAMPLES,
        "umap_n_neighbors":  UMAP_N_NEIGHBORS,
        "total_authors":     int(len(labels)),
        "n_authors":         int(len(labels)),
        "top10_clusters": [
            {"cluster_id": int(c), "size": int(s)}
            for c, s in Counter(labels[labels != -1].tolist()).most_common(10)
        ],
        "communities": communities,
        "output_files": {
            "embeddings_pt": emb_path,
            "cluster_npz":   npz_path,
        },
    }
    metrics_path = os.path.join(CLUSTER_OUTPUT_DIR, f"metrics_{year}.json")
    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=2)
    print(f"  💾 Métriques JSON    → {metrics_path}")

    # ── Visualisation ───────────────────────────────────────────
    _plot_cluster(year, z_2d, labels, n_clusters, noise_pct, sil, db, ch, G_comm)

    del z_np, z_2d, labels, clusterer, reducer
    gc.collect()
    return metrics


# ── 4. Visualisation 4 panneaux ────────────────────────────────
def _plot_cluster(year, z_2d, labels, n_clusters, noise_pct,
                  sil, db, ch, G_comm):
    palette = make_palette(n_clusters)
    fig = plt.figure(figsize=(22, 18), facecolor="#0d1117")
    fig.suptitle(
        f"THGNN — Clustering Inférence {year}  |  {n_clusters} clusters",
        fontsize=22, color="white", fontweight="bold", y=0.97
    )
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)

    # A — UMAP scatter
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.set_facecolor("#0d1117")
    nm = labels == -1
    vm = ~nm
    ax1.scatter(z_2d[nm, 0], z_2d[nm, 1],
                c="#444", s=1.5, alpha=0.25, linewidths=0, rasterized=True)
    if vm.sum() > 0:
        ax1.scatter(z_2d[vm, 0], z_2d[vm, 1],
                    c=[palette[int(l) % len(palette)] for l in labels[vm]],
                    s=2, alpha=0.5, linewidths=0, rasterized=True)
    ax1.set_title(f"UMAP 2D — {len(labels):,} auteurs ({year})",
                  color="white", fontsize=13)
    ax1.tick_params(colors="gray")
    for sp in ax1.spines.values(): sp.set_color("#333")
    ax1.set_xlabel("UMAP-1", color="gray", fontsize=9)
    ax1.set_ylabel("UMAP-2", color="gray", fontsize=9)
    ax1.text(0.02, 0.98,
             f"Silhouette : {sil:.3f}\nDavies-B.  : {db:.3f}\n"
             f"Calinski   : {ch:.0f}\nBruit      : {noise_pct:.1f}%",
             transform=ax1.transAxes, fontsize=9, color="#aef", va="top",
             bbox=dict(facecolor="#1a2030", alpha=0.7, edgecolor="#334"))

    # B — Distribution tailles
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.set_facecolor("#0d1117")
    counts = Counter(labels[labels != -1].tolist())
    sizes  = sorted(counts.values(), reverse=True)
    ax2.bar(range(1, len(sizes)+1),
            sizes,
            color=[palette[i % len(palette)] for i in range(len(sizes))],
            alpha=0.85, width=0.8)
    ax2.axhline(MIN_CLUSTER_SIZE, color="#f55", linestyle="--", linewidth=1,
                label=f"min_cluster_size={MIN_CLUSTER_SIZE}")
    ax2.set_title("Distribution tailles clusters", color="white", fontsize=13)
    ax2.set_xlabel("Clusters (triés)", color="gray", fontsize=9)
    ax2.set_ylabel("Nb auteurs", color="gray", fontsize=9)
    ax2.tick_params(colors="gray")
    for sp in ax2.spines.values(): sp.set_color("#333")
    ax2.legend(fontsize=8, facecolor="#1a2030", edgecolor="#444", labelcolor="white")

    # C — Graphe communautés
    ax3 = fig.add_subplot(gs[1, 0])
    ax3.set_facecolor("#0d1117")
    if G_comm is not None and G_comm.number_of_nodes() > 0:
        pos    = nx.spring_layout(G_comm, seed=SEED, k=2.5)
        nc     = [palette[i % len(palette)] for i in range(G_comm.number_of_nodes())]
        szs    = [max(50, G_comm.nodes[n]["size"] / 10) for n in G_comm.nodes()]
        edges  = list(G_comm.edges(data=True))
        widths = [e[2].get("weight", 0.3) * 3 for e in edges]
        nx.draw_networkx_edges(G_comm, pos, ax=ax3, width=widths,
                               edge_color="#557799", alpha=0.5)
        nx.draw_networkx_nodes(G_comm, pos, ax=ax3, node_size=szs,
                               node_color=nc, alpha=0.85)
        if G_comm.number_of_nodes() <= 30:
            nx.draw_networkx_labels(G_comm, pos, ax=ax3,
                                    font_size=7, font_color="white")
    ax3.set_title("Graphe de communautés", color="white", fontsize=13)
    ax3.axis("off")

    # D — Top-10 barres horizontales
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.set_facecolor("#0d1117")
    top10   = counts.most_common(10)
    cids    = [f"C{c}" for c, _ in top10][::-1]
    csizes  = [s for _, s in top10][::-1]
    bcolors = [palette[int(c) % len(palette)] for c, _ in reversed(top10)]
    bars = ax4.barh(cids, csizes, color=bcolors, alpha=0.85)
    for bar, val in zip(bars, csizes):
        ax4.text(bar.get_width() + max(csizes)*0.01,
                 bar.get_y() + bar.get_height()/2,
                 f"{val:,}", va="center", ha="left", color="white", fontsize=8)
    ax4.set_title("Top-10 clusters", color="white", fontsize=13)
    ax4.set_xlabel("Nb auteurs", color="gray", fontsize=9)
    ax4.tick_params(colors="gray")
    for sp in ax4.spines.values(): sp.set_color("#333")

    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig_path = os.path.join(CLUSTER_OUTPUT_DIR, f"clustering_{year}.png")
    plt.savefig(fig_path, dpi=150, bbox_inches="tight", facecolor="#0d1117")
    plt.show()
    plt.close(fig)
    print(f"  🖼️  Figure           → {fig_path}")


# ── 5. Comparaison 2024 vs 2025 ────────────────────────────────
def plot_comparison(all_metrics):
    if len(all_metrics) < 2:
        return
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor="#0d1117")
    fig.suptitle("Comparaison 2024 vs 2025 — Clustering THGNN",
                 color="white", fontsize=16, fontweight="bold")
    keys   = ["n_clusters", "silhouette", "noise_pct"]
    titles = ["Nb clusters", "Silhouette ↑", "Bruit (%)"]
    colors = ["#4ef", "#7f7", "#f84"]
    for ax, key, title, color in zip(axes, keys, titles, colors):
        ax.set_facecolor("#0d1117")
        years = [m["year"] for m in all_metrics]
        vals  = [m[key]   for m in all_metrics]
        bars  = ax.bar([str(y) for y in years], vals,
                       color=color, alpha=0.8, width=0.4)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + max(vals)*0.02,
                    f"{val:.3g}", ha="center", color="white",
                    fontsize=12, fontweight="bold")
        ax.set_title(title, color="white", fontsize=13)
        ax.tick_params(colors="gray")
        for sp in ax.spines.values(): sp.set_color("#333")
        ax.set_ylim(0, max(vals) * 1.25)
    fig.tight_layout()
    cmp_path = os.path.join(CLUSTER_OUTPUT_DIR, "comparaison_2024_2025.png")
    plt.savefig(cmp_path, dpi=150, bbox_inches="tight", facecolor="#0d1117")
    plt.show()
    plt.close(fig)
    print(f"  🖼️  Comparaison      → {cmp_path}")


# ── 6. Boucle principale ───────────────────────────────────────
print("\n" + "🔬 " * 28)
print("CLUSTERING INFÉRENCE 2024 & 2025")
print("🔬 " * 28)
print(f"\nOrdre de traitement : {CLUSTER_YEARS}")
print("→ 2024 : prior_state = états GRU fin 2023 (depuis checkpoint)")
print("→ 2025 : prior_state = états GRU fin 2024 (mis à jour après inférence 2024)\n")
print("→ Pipeline : Inférence → HDBSCAN (128-dim, cosine) → UMAP (visu)\n")

all_metrics_list = []

for year in CLUSTER_YEARS:
    m = cluster_year_inference(year, model_clust, author_state_bank_clust)
    if m is not None:
        all_metrics_list.append(m)

# ── Résumé terminal ────────────────────────────────────────────
print("\n" + "="*70)
print("📊 RÉSUMÉ GLOBAL")
print("="*70)
print(f"{'Année':<8} {'Clusters':>10} {'Bruit%':>8} {'Silhouette':>12} "
      f"{'DB':>10} {'CH':>12}")
print("-"*65)
for m in all_metrics_list:
    print(f"{m['year']:<8} {m['n_clusters']:>10} {m['noise_pct']:>7.1f}% "
          f"{m['silhouette']:>12.4f} {m['davies_bouldin']:>10.4f} "
          f"{m['calinski_harabasz']:>12.1f}")

plot_comparison(all_metrics_list)

# ── JSON global ───────────────────────────────────────────────
summary_path = os.path.join(CLUSTER_OUTPUT_DIR, "clustering_summary.json")
with open(summary_path, "w") as f:
    json.dump({
        "checkpoint": CKPT_PATH,
        "years":      [m["year"] for m in all_metrics_list],
        "params": {
            "min_cluster_size": MIN_CLUSTER_SIZE,
            "min_samples":      MIN_SAMPLES,
            "umap_n_neighbors": UMAP_N_NEIGHBORS,
            "umap_metric":      UMAP_METRIC,
            "hdbscan_metric":   "cosine",
            "note":             "HDBSCAN sur embeddings 128-dim, UMAP pour visualisation uniquement",
        },
        "metrics": all_metrics_list,
    }, f, indent=2)

print(f"\n✅ Terminé !")
print(f"\n📂 Fichiers générés dans {CLUSTER_OUTPUT_DIR}/")
for m in all_metrics_list:
    y = m["year"]
    n = m["n_authors"]
    d = HIDDEN_DIM
    print(f"   embeddings_{y}.pt   ← Tensor ({n:,} × {d})  embeddings bruts")
    print(f"   cluster_{y}.npz     ← z_2d ({n:,} × 2) + labels ({n:,},) + node_ids")
    print(f"   metrics_{y}.json    ← métriques + top10 + communautés")
    print(f"   clustering_{y}.png  ← figure 4 panneaux")
print(f"   comparaison_2024_2025.png")
print(f"   clustering_summary.json")

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
💡 Recharger les embeddings dans une autre cellule :

  d = torch.load("/kaggle/working/clustering/embeddings_2024.pt")
  emb      = d["embeddings"]   # Tensor (N, 128)
  node_ids = d["node_ids"]     # Tensor (N,)
  g_idx    = d["global_idx"]   # Tensor (N,)

  npz    = np.load("/kaggle/working/clustering/cluster_2024.npz")
  z_2d   = npz["z_2d"]    # (N, 2)  coordonnées UMAP (visu)
  labels = npz["labels"]  # (N,)    cluster ID (-1 = bruit HDBSCAN)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

In [ ]:
import numpy as np
import json
from collections import defaultdict, Counter
import pandas as pd

# ============================================================
# 1. LOAD PAPER EMBEDDINGS (SAFE + NORMALIZED)
# ============================================================

path = "/kaggle/input/datasets/ayamhiri/temporal-graph-dataset/embeddings/paper_embeddings.npy"

paper_emb = np.fromfile(path, dtype=np.float32).reshape(-1, 384)

paper_emb = paper_emb / (np.linalg.norm(paper_emb, axis=1, keepdims=True) + 1e-9)

print("Paper embeddings:", paper_emb.shape)

# ============================================================
# 2. LOAD CLUSTERS
# ============================================================

cluster_data = np.load(
    "/kaggle/input/datasets/ayamhiri/output-clustering-classic/cluster_2024.npz"
)

labels = cluster_data["labels"]
author_ids = cluster_data["node_ids"]

print("Clusters:", len(set(labels)) - (1 if -1 in labels else 0))

# ============================================================
# 3. AUTHOR → PAPERS
# ============================================================

edge_path = "/kaggle/input/datasets/ayamhiri/temporal-graph-dataset/edges/edges_author_paper.jsonl"

author_to_papers = defaultdict(list)

with open(edge_path, "r") as f:
    for line in f:
        e = json.loads(line)
        author_to_papers[int(e["src"])].append(int(e["dst"]))

# ============================================================
# 4. PAPER METADATA
# ============================================================

paper_path = "/kaggle/input/datasets/ayamhiri/temporal-graph-dataset/nodes/nodes_papers.jsonl"

paper_db = {}

with open(paper_path, "r") as f:
    for line in f:
        p = json.loads(line)

        pid = p.get("p_idx")
        try:
            pid = int(pid)
        except:
            continue

        paper_db[pid] = {
            "title": p.get("title", ""),
            "keywords": p.get("keywords", [])
        }

# ============================================================
# 5. BUILD CLUSTERS → PAPERS
# ============================================================

cluster_papers = defaultdict(set)

for author, label in zip(author_ids, labels):

    if label == -1:
        continue

    papers = author_to_papers.get(int(author), [])

    for p in papers:
        if 0 <= p < len(paper_emb):
            cluster_papers[int(label)].add(int(p))

# ============================================================
# 6. BUILD CLUSTER REPRESENTATION
# ============================================================

cluster_labels = {}

for c, papers in cluster_papers.items():

    papers = list(papers)

    if len(papers) == 0:
        continue

    vecs = paper_emb[papers]

    centroid = vecs.mean(axis=0)
    centroid = centroid / (np.linalg.norm(centroid) + 1e-9)

    sims = vecs @ centroid
    top_idx = np.argsort(sims)[-10:][::-1]
    top_papers = [papers[i] for i in top_idx]

    keywords, titles = [], []

    for p in top_papers:
        meta = paper_db.get(int(p))
        if meta:
            keywords += meta.get("keywords", [])
            titles.append(meta.get("title", ""))

    top_keywords = Counter(keywords).most_common(5)

    cluster_labels[int(c)] = {
        "label": " / ".join([k for k, _ in top_keywords]) or f"Cluster {c}",
        "top_keywords": [(str(k), int(v)) for k, v in top_keywords],
        "top_titles": [str(t) for t in titles[:5]],
        "size": int(len(papers))
    }

# ============================================================
# 7. SAFE JSON EXPORT (FIXED)
# ============================================================

def make_json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(x) for x in obj]
    if isinstance(obj, tuple):
        return list(obj)
    if hasattr(obj, "item"):
        return obj.item()
    return obj

output_json = "/kaggle/working/cluster_labels_2024.json"

with open(output_json, "w") as f:
    json.dump(make_json_safe(cluster_labels), f, indent=2)

print("Saved JSON:", output_json)

# ============================================================
# 8. CSV SUMMARY (FIXED TYPES)
# ============================================================

rows = []

for c, v in cluster_labels.items():

    rows.append({
        "cluster": int(c),
        "label": v["label"],
        "size": int(v["size"]),
        "top_keywords": ", ".join([k for k, _ in v["top_keywords"]]),
        "top_titles": " || ".join(v["top_titles"])
    })

df = pd.DataFrame(rows)

csv_path = "/kaggle/working/cluster_summary_2024.csv"
df.to_csv(csv_path, index=False)

print("Saved CSV:", csv_path)